# Domain knowledge at scale 

In [2]:
import pandas as pd
import numpy as np
import math
import collections
from transformers import AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling, default_data_collator, TrainingArguments, Trainer, pipeline
import torch
from datasets import Dataset

import sys
import os 

sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks
from comments_saver import CommentsSaver

/opt/miniconda3/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Set up basic model parameters

In [3]:
model_checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

nlp = NLP_Tasks(model_checkpoint)

Device set to use mps:0


In [4]:
text = "This is an awful [MASK]."

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits

# Find the location of [MASK] in the input and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]

# Pick the 5 [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(text.replace("[MASK]", tokenizer.decode([token])))

This is an awful sight.
This is an awful thing.
This is an awful idea.
This is an awful coincidence.
This is an awful mess.


Load the data

In [6]:
cs = CommentsSaver()
df = cs.read_all()

Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.


In [7]:
# drop any rows of data with no text
df = df.dropna(subset=["comment_text"])

# drop any rows of data which aren't strings
df = df[df["comment_text"].apply(lambda x: isinstance(x, str))]

In [8]:
len(df)

16565

In [9]:
df.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date
0,17544,Ealing,223203FUL_111,223203FUL,80 SHAKESPEARE ROAD LONDON W3 6SN W3 6SN,Objects,2022-08-25,Acton High Street is considered to be one of t...,2025-03-15
1,17545,Ealing,223203FUL_112,223203FUL,34 lexden road london w3 9nz London W3 9NZ W3 9NZ,Objects,2022-08-25,Taking away trees .enough of highrise flats in...,2025-03-15
2,17546,Ealing,223203FUL_113,223203FUL,3 Derwentwater Road London W3 6DE W3 6DE,Objects,2022-08-25,The proposal is so excessive for the plot. Twe...,2025-03-15
3,17547,Ealing,223203FUL_114,223203FUL,25 Rosemont Road LONDON W3 9LU W3 9LU,Objects,2022-08-25,I object to this proposal.\nI have lived in Ac...,2025-03-15
4,17548,Ealing,223203FUL_115,223203FUL,41 Lexden Road London W3 9NY W3 9NY,Objects,2022-08-24,I object to this development because of the fo...,2025-03-15


In [15]:
# Convert DataFrame to Hugging Face Dataset
data = Dataset.from_pandas(df)

In [16]:
len(data)

16565

In [26]:
# Tokenization function
def tokenize_function(examples):
    result = tokenizer(examples["comment_text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result

# Use batched=True to activate fast multithreading!
tokenized_datasets = data.map(tokenize_function, batched=True, remove_columns=["address", "stance", "date", "comment_text"])

tokenized_datasets

Map: 100%|██████████| 16565/16565 [00:02<00:00, 6704.53 examples/s]


Dataset({
    features: ['id', 'council', 'comment_id', 'application_id', 'add_date', 'input_ids', 'attention_mask', 'word_ids'],
    num_rows: 16565
})

In [45]:
def group_texts(examples):
    chunk_size = 128
    
    # Flatten data
    concatenated_examples = {k: [] for k in examples.keys()}
    for k in examples.keys():
        for item in examples[k]:
            if isinstance(item, list):  
                concatenated_examples[k].extend(item)
            else:
                concatenated_examples[k].append(item)

    # Check how many tokens we have before chunking
    total_length = len(concatenated_examples["input_ids"])
    print(f"Total tokens before chunking: {total_length}")

    # Compute total usable length for chunking
    total_length = (total_length // chunk_size) * chunk_size
    print(f"Usable tokens (after dropping remainder): {total_length}")

    # Chunking
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }

    # Print how many rows are created after chunking
    print(f"New dataset size after chunking: {len(result['input_ids'])}")

    if "input_ids" in result:
        result["labels"] = result["input_ids"].copy()

    return result



In [40]:
tokenized_datasets

Dataset({
    features: ['id', 'council', 'comment_id', 'application_id', 'add_date', 'input_ids', 'attention_mask', 'word_ids'],
    num_rows: 16565
})

In [46]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)

Map:   6%|▌         | 1000/16565 [00:00<00:04, 3771.62 examples/s]

Total tokens before chunking: 312177
Usable tokens (after dropping remainder): 312064
New dataset size after chunking: 2438


Map:  12%|█▏        | 2000/16565 [00:00<00:03, 4150.72 examples/s]

Total tokens before chunking: 289506
Usable tokens (after dropping remainder): 289408
New dataset size after chunking: 2261
Total tokens before chunking: 218472
Usable tokens (after dropping remainder): 218368
New dataset size after chunking: 1706


Map:  24%|██▍       | 4000/16565 [00:00<00:02, 4530.67 examples/s]

...

Map:  78%|███████▊  | 13000/16565 [00:02<00:00, 5709.76 examples/s]

Total tokens before chunking: 151504
Usable tokens (after dropping remainder): 151424
New dataset size after chunking: 1183
Total tokens before chunking: 92148
Usable tokens (after dropping remainder): 92032
New dataset size after chunking: 719
Total tokens before chunking: 200334
Usable tokens (after dropping remainder): 200320


Map:  97%|█████████▋| 16000/16565 [00:02<00:00, 5979.27 examples/s]

New dataset size after chunking: 1565
Total tokens before chunking: 194575
Usable tokens (after dropping remainder): 194560
New dataset size after chunking: 1520
Total tokens before chunking: 96499
Usable tokens (after dropping remainder): 96384
New dataset size after chunking: 753


Map: 100%|██████████| 16565/16565 [00:03<00:00, 5400.88 examples/s]


In [47]:
lm_datasets

Dataset({
    features: ['id', 'council', 'comment_id', 'application_id', 'add_date', 'input_ids', 'attention_mask', 'word_ids', 'labels'],
    num_rows: 27030
})

In [48]:
downsampled_dataset = lm_datasets.train_test_split(test_size=0.2)

In [ ]:
# downsampled_dataset = nlp.process_data(data)

Map:   0%|          | 0/16565 [00:00<?, ? examples/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Token indices sequence length is longer than the specified maximum sequence length for this model (1000 > 512). Running this sequence through the model will result in indexing errors
Map: 100%|██████████| 16565/16565 [00:02<00:00, 7792.87 examples/s]


In [49]:
# size of the training dataset
len(downsampled_dataset["train"])

21624

Now to train!

In [50]:
# mlm is the fraction of tokens to mask - 15% is popular in the literature. 
mlm_prob = 0.15

# mask the tokens
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=mlm_prob)

In [51]:
batch_size = 8

logging_steps = len(downsampled_dataset["train"]) // batch_size
training_args = TrainingArguments(
    output_dir="../outputs",
    overwrite_output_dir=True,
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    logging_steps=logging_steps,
    fp16=False,
    bf16=True # Note this enables bfloat16 conversion which is supported by the Apple MPD backend
)

In [52]:
trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer 
)

/var/folders/4n/x6w1yfcx01qbymrsfpz4ybq00000gn/T/ipykernel_34237/1257576995.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [53]:
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 17.04


In [54]:
# run the training loop!
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

See how the model performs on some masked sentences

In [ ]:
mask_filler = pipeline("fill-mask", model=model, tokenizer=tokenizer)

preds = mask_filler(text)
for pred in preds:
    print(f"{pred['sequence']}")

Device set to use mps:0


this is an awful thing.
this is an awful sight.
this is an awful place.
this is an awful idea.
this is an awful job.


Save the model

In [ ]:
# model.save_pretrained("../outputs/nlp_fine_tuning/distilbert-base-uncased_comments")
# tokenizer.save_pretrained("../outputs/nlp_fine_tuning/distilbert-base-uncased_comments")